# Notebook 01 - Fintech Daily Bars Extraction/Backfill

## Purpose

This notebook guides a Google Colab runtime through the Fintech daily bars extraction/backfill workflow. It initializes a local Fintech workspace, preserves the active `SESSION_ID`, runs native Fintech command examples, and documents how generated daily bars can be reviewed and persisted.

## Upstream apps used

- `fintech-market-ingestion`: primary upstream app for project initialization, daily bars extraction/backfill, session-save previews, and archive/backup command previews.
- `stratlake-trade-engine`: not used in this notebook.

## Workflow role

- extraction/backfill
- validation
- session persistence preview
- archive preview

## Runtime assumptions

- Designed for Google Colab.
- Active runtime work happens under `/content`.
- Google Drive is used only for persistence, backup, archive, and restore storage.
- Generated Parquet/data files, archives, restore packs, local workspaces, runtime folders, notebook outputs, and secrets are not committed.

## Required user inputs

- `<DRIVE_FOLDER_PLACEHOLDER>`: Google Drive folder used for tutorial persistence storage.
- `ALPACA_API_KEY_ID`: Colab Secret name or hidden prompt input for the Alpaca API key id.
- `ALPACA_API_SECRET_KEY`: Colab Secret name or hidden prompt input for the Alpaca API secret key.
- `SESSION_ID`: generated by `fintech-init-project` and reused by later commands.

## Secrets

This notebook must use Colab Secrets or safe hidden prompts for credentials.

Do not hard-code API keys, tokens, secrets, `.env` values, credential JSON, or private keys.

## Path conventions

Use portable runtime variables. Do not hard-code local machine paths, personal Google Drive paths, or committed local workspace paths.

- `WORKSPACE_ROOT`: active Fintech workspace under `/content`.
- `DRIVE_ROOT`: user-selected Google Drive persistence root placeholder.
- `SESSION_ID`: active notebook/project session id generated at runtime.
- `ARCHIVE_ID`: archive identifier derived from `SESSION_ID` when archive previews are used.

## Native-command-first boundary

This notebook orchestrates upstream commands, validates expected runtime files, previews persistence/archive commands, and supports human review. It does not reimplement Fintech ingestion, extraction, normalization, archive, restore, or artifact logic.

## Generated artifact boundaries

Runtime-generated outputs remain outside Git:

- Daily bars Parquet/data under the local `/content` Fintech workspace.
- Session-save payloads under the selected Google Drive persistence root.
- Archive backup packs under the selected Google Drive backup root.
- Restore outputs copied back into the local `/content` workspace.

Local repository tests must not run live ingestion, mount Google Drive, prompt for credentials, write archives, restore data, or call live APIs. Manual Colab smoke testing remains the final runtime confirmation step.

## Validation before commit

Run:

```bash
python scripts/scan_for_secret_patterns.py .
python scripts/check_notebooks_no_outputs.py notebooks
python scripts/validate_repo_cleanliness.py .
python scripts/validate_notebook_execution_readiness.py notebooks/01_fintech_daily_bars_extraction_backfill.ipynb --config config/notebook_test.toml
```


## Tutorial Context

This notebook follows Notebook 00 and focuses on the first controlled extraction workflow in the series.

Planned sequence:

```text
Notebook 00 - Setup and Storage Overview
Notebook 01 - Fintech Daily Bars Extraction/Backfill
Notebook 02 - Session Save and Restore
Notebook 03 - Archive Backup Pack and Restore
```

The active local workspace is under `/content`. Google Drive is referenced through a placeholder persistence root and is not the active app workspace.

The key convention preserved in this notebook is that the active project session determines `SESSION_ID`, and later paths and commands interpolate `{SESSION_ID}` directly.


## 1. Install Required Packages

Manual Colab/runtime-only setup cell.

Run this cell in a fresh Colab runtime when preparing the notebook session. It installs notebook runtime dependencies and the upstream `fintech-market-ingestion` package. Local repository tests must not run package installation or network access.


In [ ]:
!python -m pip install --upgrade pip
!python -m pip install "pandas-market-calendars>=5.0"
!python -m pip install -i https://test.pypi.org/simple/ fintech-market-ingestion


## 2. Verify Required CLI Commands

Safe command-discovery cell.

This notebook expects the native upstream CLI surface:

```text
fintech-init-project
fintech-backfill-daily
fintech-save-session
fintech-backup-data
```

If a command is missing in Colab, rerun the installation cell. If a command is missing locally during repository validation, that should be treated as an expected missing-upstream-command warning where configured.


In [ ]:
import shutil

required_commands = [
    "fintech-init-project",
    "fintech-backfill-daily",
    "fintech-save-session",
    "fintech-backup-data",
]

for command in required_commands:
    command_path = shutil.which(command)
    print(f"{command}: {command_path if command_path else 'NOT FOUND'}")


## 3. Optional: Inspect Deployed CLI Help

Help-only command cell.

These commands display the installed upstream CLI help surface. They are safe command previews, but local validation may skip help checks when the upstream app is not installed.


In [ ]:
!fintech-init-project --help
!fintech-backfill-daily --help
!fintech-save-session --help
!fintech-backup-data --help


## 4. Authorize and Mount Google Drive

Manual Colab/runtime-only cell.

Run this only in Google Colab when you intentionally want persistence, backup, archive, or restore storage. Local repository tests must not mount Google Drive.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 5. Define Local and Drive Roots

The local workspace is the active runtime workspace under `/content`.

`DRIVE_ROOT` is a placeholder for user-selected Google Drive persistence storage. It is used only for session saves, backups, archives, and restore storage, not as the active app workspace.


In [ ]:
from pathlib import Path
import json
import os
import getpass

WORKSPACE_ROOT = Path("/content/fintech-market-ingestion-demo")
DRIVE_ROOT = Path("/content/drive/MyDrive/<DRIVE_FOLDER_PLACEHOLDER>")
DRIVE_PROJECT_ROOT = DRIVE_ROOT / "fintech-market-ingestion"

CONFIGS_ROOT = WORKSPACE_ROOT / "configs"
SOURCE_DATASET_ROOT = WORKSPACE_ROOT / "data" / "curated"
DAILY_BARS_ROOT = SOURCE_DATASET_ROOT / "bars_daily"
TICKERS_FILE = CONFIGS_ROOT / "tickers_sample.txt"

print("Workspace root:", WORKSPACE_ROOT)
print("Drive persistence root:", DRIVE_PROJECT_ROOT)
print("Daily bars output:", DAILY_BARS_ROOT)
print("Tickers file:", TICKERS_FILE)


## 6. Initialize the Project Session

Manual Colab/runtime-only command cell.

A standalone extraction notebook should initialize a local project workspace before running the backfill. This command creates workspace folders and session metadata under the local `/content` workspace. It does not run ingestion, save to Drive, or create archive backups.


In [ ]:
!fintech-init-project \
  --root /content/fintech-market-ingestion-demo \
  --notebooks \
  --with-session \
  --session-name extraction_daily_bars_demo


## 7. Extract `SESSION_ID` From the Initialized Notebook Session

Runtime metadata parsing cell.

This cell reads the latest local project-session manifest and extracts the active `SESSION_ID`. Later shell commands use `{SESSION_ID}` directly so session save, archive backup, and restore previews stay aligned to the same notebook-session identity.


In [ ]:
session_manifest_paths = sorted(
    (WORKSPACE_ROOT / "artifacts" / "sessions").glob("*/session_manifest.json"),
    key=lambda path: path.stat().st_mtime,
)

if not session_manifest_paths:
    raise FileNotFoundError("No session_manifest.json files found. Run fintech-init-project first.")

SESSION_MANIFEST_PATH = session_manifest_paths[-1]

with SESSION_MANIFEST_PATH.open("r", encoding="utf-8") as file:
    SESSION_MANIFEST = json.load(file)

SESSION_ID = SESSION_MANIFEST["session_id"]

print("Session manifest:", SESSION_MANIFEST_PATH)
print("SESSION_ID:", SESSION_ID)


## 8. Prepare `{SESSION_ID}`-Based Google Drive Directories

Manual Colab/runtime-only persistence cell.

This cell creates Drive directories for the active `SESSION_ID`. The session folder is used for lightweight session save/restore workflows. The backup folder is used for archive backup packs created from larger local curated datasets.

Google Drive remains persistence storage only.


In [ ]:
DRIVE_SESSION_ROOT = DRIVE_PROJECT_ROOT / "sessions" / SESSION_ID
DRIVE_BACKUP_ROOT = DRIVE_SESSION_ROOT / "backups"

ARCHIVE_ID = f"daily-bars-{SESSION_ID}"

DRIVE_SESSION_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

WORKSPACE_ROOT_STR = WORKSPACE_ROOT.as_posix()
SOURCE_DATASET_ROOT_STR = SOURCE_DATASET_ROOT.as_posix()
DRIVE_SESSION_ROOT_STR = DRIVE_SESSION_ROOT.as_posix()
DRIVE_BACKUP_ROOT_STR = DRIVE_BACKUP_ROOT.as_posix()
DAILY_BARS_ROOT_STR = DAILY_BARS_ROOT.as_posix()
TICKERS_FILE_STR = TICKERS_FILE.as_posix()

print("SESSION_ID:", SESSION_ID)
print("Archive ID:", ARCHIVE_ID)
print("Drive session root exists:", DRIVE_SESSION_ROOT.exists())
print("Drive session root:", DRIVE_SESSION_ROOT)
print("Drive backup root exists:", DRIVE_BACKUP_ROOT.exists())
print("Drive backup root:", DRIVE_BACKUP_ROOT)


## 9. Optional: Find a Previous Drive Session

Manual Colab/runtime-only persistence cell.

Most extraction runs should use the current `SESSION_ID`. If you intentionally want to reference a previous Drive session, list available Drive sessions here and set `RESTORE_SESSION_ID` to one of them.

This notebook does not restore by default. Restore-focused notebooks use this pattern when they need to recover saved state or archived data.


In [ ]:
available_drive_sessions = sorted(
    path.name for path in (DRIVE_PROJECT_ROOT / "sessions").glob("*") if path.is_dir()
)

print("Available Drive session IDs:")
for session_id in available_drive_sessions:
    print(" -", session_id)

RESTORE_SESSION_ID = SESSION_ID
RESTORE_SESSION_ROOT = DRIVE_PROJECT_ROOT / "sessions" / RESTORE_SESSION_ID
RESTORE_BACKUP_ROOT = RESTORE_SESSION_ROOT / "backups"
RESTORE_ARCHIVE_ID = f"daily-bars-{RESTORE_SESSION_ID}"

print("Using RESTORE_SESSION_ID:", RESTORE_SESSION_ID)
print("Restore archive ID:", RESTORE_ARCHIVE_ID)
print("Restore session root exists:", RESTORE_SESSION_ROOT.exists())
print("Restore backup root exists:", RESTORE_BACKUP_ROOT.exists())


## 10. Configure Alpaca API Environment Variables

Manual Colab/runtime-only credential cell.

The daily backfill command reads Alpaca credentials from environment variables. In Colab, prefer storing credentials in Colab Secrets. This cell tries Colab Secrets and then falls back to hidden prompts.

Required secret names:

```text
ALPACA_API_KEY_ID
ALPACA_API_SECRET_KEY
```

Optional runtime defaults:

```text
ALPACA_DATA_BASE_URL=https://data.alpaca.markets
ALPACA_FEED=iex
```

Do not print, store, or commit credential values.


In [ ]:
try:
    from google.colab import userdata
except ImportError:
    userdata = None

api_key_id_name = "ALPACA_API_KEY_ID"
api_private_name = "ALPACA_API_" + "SECRET_KEY"

if userdata is not None:
    api_key_id_value = userdata.get(api_key_id_name)
    api_private_value = userdata.get(api_private_name)

    if api_key_id_value:
        os.environ[api_key_id_name] = api_key_id_value
    if api_private_value:
        os.environ[api_private_name] = api_private_value

# Fallback for notebook environments without Colab Secrets.
if not os.environ.get(api_key_id_name):
    os.environ[api_key_id_name] = getpass.getpass(f"{api_key_id_name}: ")

if not os.environ.get(api_private_name):
    os.environ[api_private_name] = getpass.getpass(f"{api_private_name}: ")

os.environ.setdefault("ALPACA_DATA_BASE_URL", "https://data.alpaca.markets")
os.environ.setdefault("ALPACA_FEED", "iex")

print("Alpaca credential environment variables configured for this runtime.")
print("ALPACA_DATA_BASE_URL:", os.environ["ALPACA_DATA_BASE_URL"])
print("ALPACA_FEED:", os.environ["ALPACA_FEED"])


## 11. Prepare a Ticker File

Runtime setup cell.

The backfill command reads ticker symbols from a text file in the local `/content` workspace. This cell creates a small sample file only when one does not already exist. The ticker file is runtime configuration, not generated repository data.


In [ ]:
CONFIGS_ROOT.mkdir(parents=True, exist_ok=True)

if TICKERS_FILE.exists():
    print("Ticker file already exists:", TICKERS_FILE)
else:
    TICKERS_FILE.write_text("AAPL\nMSFT\nNVDA\n", encoding="utf-8")
    print("Created ticker file:", TICKERS_FILE)

print(TICKERS_FILE.read_text(encoding="utf-8"))


## 12. Run the Daily Bars Backfill

Manual Colab/runtime-only live API command cell.

This command extracts daily market bars from Alpaca and writes them into the local curated data folder under `/content`.

Generated Parquet/data files are runtime-only and must not be committed. Local repository tests must not run this cell.

The command includes `--source session_{SESSION_ID}` so the active session identifier is recorded in the source tag passed to the upstream normalizer. The `{SESSION_ID}` interpolation is intentionally preserved.


In [ ]:
!fintech-backfill-daily \
  --symbols {TICKERS_FILE_STR} \
  --start 2024-10-01 \
  --end 2025-04-15 \
  --out {DAILY_BARS_ROOT_STR} \
  --feed iex \
  --source session_{SESSION_ID} \
  --window month


## 13. Inspect the Local Curated Output

Manual Colab/runtime-only generated-data inspection cell.

After extraction, check that local Parquet files exist. This reviews runtime-generated files under `/content`; it must not add Parquet files, file listings, or generated data to Git.


In [ ]:
parquet_files = sorted(DAILY_BARS_ROOT.rglob("*.parquet"))

print("Daily bars root exists:", DAILY_BARS_ROOT.exists())
print("Parquet file count:", len(parquet_files))

for path in parquet_files[:20]:
    print(path)


## 14. Optional: Preview a `{SESSION_ID}` Session Save

Dry-run/preview command cell.

This is a session-save preview for lightweight workflow state and small or moderate curated outputs. The command uses `{SESSION_ID}` in both `--session-id` and the Drive destination.

For larger `data/curated/` datasets, prefer the archive backup pack cells below because they package many partitioned files into larger transfer-friendly backup artifacts.


In [ ]:
!fintech-save-session \
  --root {WORKSPACE_ROOT_STR} \
  --session-id {SESSION_ID} \
  --policy curated_daily_bars \
  --adapter google-drive \
  --destination {DRIVE_SESSION_ROOT_STR} \
  --include-curated-data \
  --dry-run


## 15. Optional: Write the Session Save After Reviewing the Dry Run

Manual Colab/runtime-only write command template.

Run this only if you intentionally want to copy selected project/session data to the `{SESSION_ID}` Drive folder as a session save. For large `data/curated/` folders, skip this and use archive backup packs instead.


In [ ]:
# Uncomment to perform the session save after reviewing the dry-run output.
# !fintech-save-session \
#   --root {WORKSPACE_ROOT_STR} \
#   --session-id {SESSION_ID} \
#   --policy curated_daily_bars \
#   --adapter google-drive \
#   --destination {DRIVE_SESSION_ROOT_STR} \
#   --include-curated-data


## 16. Optional: Preview a `{SESSION_ID}` Archive Backup Pack

Dry-run/preview command cell.

Archive backup packs are useful for larger curated datasets because Google Drive can be slow when copying many small partitioned Parquet files. This dry run plans an archive pack from local curated data into the active session's Drive backup folder.

The archive identity is derived from `{SESSION_ID}`:

```text
ARCHIVE_ID = daily-bars-{SESSION_ID}
```


In [ ]:
!fintech-backup-data pack \
  --workspace-root {WORKSPACE_ROOT_STR} \
  --source-dataset-root {SOURCE_DATASET_ROOT_STR} \
  --backup-root {DRIVE_BACKUP_ROOT_STR} \
  --backup-id {ARCHIVE_ID} \
  --shard-size-mb 512 \
  --dry-run


## 17. Optional: Create the Archive Backup Pack

Manual Colab/runtime-only write command template.

Run this only after reviewing the dry run. This writes archive-pack outputs to the `{SESSION_ID}` Drive backup folder. The active dataset remains local; the archive is a persistence artifact.


In [ ]:
# Uncomment to create the archive backup pack after reviewing the dry run.
# !fintech-backup-data pack \
#   --workspace-root {WORKSPACE_ROOT_STR} \
#   --source-dataset-root {SOURCE_DATASET_ROOT_STR} \
#   --backup-root {DRIVE_BACKUP_ROOT_STR} \
#   --backup-id {ARCHIVE_ID} \
#   --shard-size-mb 512


## 18. Optional: Validate the Archive Backup Pack

Manual Colab/runtime-only archive validation command template.

After creating a real archive pack, validate it before relying on it for restore. This command is intentionally tied to `{ARCHIVE_ID}`, which is derived from `{SESSION_ID}`.


In [ ]:
# Uncomment after creating the archive pack.
# !fintech-backup-data validate \
#   --backup-root {DRIVE_BACKUP_ROOT_STR} \
#   --backup-id {ARCHIVE_ID}


## 19. Restore-Readiness Command Template

Preview-only command template.

This extraction notebook does not restore by default. The restore pattern should copy archived curated data back into the local `/content` workspace before downstream notebooks run. This keeps Google Drive as persistence storage and keeps local notebook storage as the active execution layer.

The command below prints the restore template using the active `{SESSION_ID}`-derived archive. It does not execute restore.


In [ ]:
restore_command = (
    "fintech-backup-data restore \\\n"
    f"  --workspace-root {WORKSPACE_ROOT_STR} \\\n"
    f"  --target-dataset-root {SOURCE_DATASET_ROOT_STR} \\\n"
    f"  --backup-root {DRIVE_BACKUP_ROOT_STR} \\\n"
    f"  --backup-id {ARCHIVE_ID}"
)

print(restore_command)


## 20. Storage Boundary

This notebook is about extraction.

The intended storage model is:

```text
Alpaca API
  -> local notebook runtime
  -> data/curated/bars_daily
  -> optional SESSION_ID session save
  -> optional SESSION_ID archive backup pack
```

Use session saving when you want to preserve lightweight workflow state or a small selected dataset.

Use archive backup packs when the curated dataset is large or contains many partitioned files.

Google Drive is persistence storage only. It should not be used as the active Parquet working root for performance-sensitive workflows.


## Notebook Summary

In this notebook, you prepared a manual Colab workflow to:

- install standalone runtime dependencies
- install `fintech-market-ingestion`
- verify the expected notebook/storage CLIs
- authorize and mount Google Drive for persistence storage
- initialize a local project session under `/content`
- extract and preserve `SESSION_ID`
- create `{SESSION_ID}`-based Drive session and backup directories
- configure Alpaca API environment variables through safe runtime mechanisms
- prepare a ticker file
- run a native Fintech daily bars backfill in Colab
- inspect local curated output
- preview a `{SESSION_ID}` session save
- preview a `{SESSION_ID}` archive backup pack
- print a restore-readiness command template

Before committing this notebook, clear all outputs, keep execution counts null, review raw JSON for paths and secrets, and run repository validation. Manual Colab smoke testing remains a later runtime confirmation step.
